In [11]:
#!conda install -c conda-forge gfortran_linux-64

Channels:
 - conda-forge
Platform: linux-64

NoWritablePkgsDirError: No writeable pkgs directories configured.
  - /opt/conda/pkgs
  - /home/jovyan/.conda/pkgs



In [12]:
!pip install git+https://github.com/dfm/python-fsps.git

Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/dfm/python-fsps.git to /tmp/pip-req-build-quernrb1
  Running command git clone --filter=blob:none --quiet https://github.com/dfm/python-fsps.git /tmp/pip-req-build-quernrb1
  Resolved https://github.com/dfm/python-fsps.git to commit 33c2989ec57ddb0f0a0f12b6fd5360394729dd23
  Running command git submodule update --init --recursive -q
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for fsps (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [23 lines of output]
      *** scikit-build-core 0.11.6 using CMake 4.1.0 (wheel)
      *** Configuring CMake...
      loading initial cache file /tmp/tmp31miuyqd/build/CMakeInit.txt
      -- The C compiler identification is GNU 13.2.0
      -- 

In [1]:
#!prospector my_script.py

In [2]:
#!pip uninstall sedpy -y
#!pip install git+https://github.com/bd-j/sedpy.git

In [3]:
import numpy as np
from prospect.models import priors, sedmodel
from prospect.sources import CSPSpecBasis
from sedpy.observate import load_filters
import prospect.io.read_results as reader
from prospect.fitting import fit_model
from dynesty.dynamicsampler import stopping_function, weight_function

from astropy.table import QTable, Table, Column

try:
    from sedpy.observate import Filter
except ImportError:
    from sedpy import observate
    Filter = observate.Filter

In [4]:
import sedpy
print(sedpy.__version__)
print(sedpy.__file__)

0.4.2.dev3+g303727a90
/home/jovyan/.local/lib/python3.12/site-packages/sedpy/__init__.py


In [5]:
c = 299792.5 #km s^-1
H_0 = 70 #km s^-1 MpC^-1

"""
These are the rest wavelengths in angstroms for the lines that are important for the fits.
"""
HeII_2733 = 2733.289
MgII_2800 = 2799.000
FeIV_2829 = 2829.36
FeIV_2836 = 2835.740
ArIV_2854 = 2853.670
ArIV_2868 = 2868.210

# Pivot Bands wavelength
G_Band_Wave = 4791.155332473396 #AA https://prc.nao.ac.jp/citizen-science/hscv/hscdata_e.html - These are turned into Pivot Wavelengths by Bagpipes
R_Band_Wave = 7688.726218179237
I_Band_Wave = 6214.060657315306
Z_Band_Wave = 8903.034519679457
Y_Band_Wave = 9771.696773766653




"""
To get the FWHM to use in my MgII fitting in velocity space. 
This will give a minumum not a maximum.
FWHM = speed of light / spectral resolution.
"""
R = 800 #Spectral Resolution

FWHM_Velocity = c/R


# Your existing code should work better now
hsc_filter_files = {
    'hsc_g': '/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/hsc_g.dat',
    'hsc_r': '/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/hsc_r.dat', 
    'hsc_i': '/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/hsc_i.dat',
    'hsc_z': '/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/hsc_z.dat',
    'hsc_y': '/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/hsc_y.dat'
}

In [6]:
SED_File = Table.read("/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/Table_Gal_Flux_MicroJansky_For_Alexa.fits", format = 'fits')

G_Bandpass_Filter_Curve = np.loadtxt("/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/hsc_g.dat")
R_Bandpass_Filter_Curve = np.loadtxt("/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/hsc_r.dat")
I_Bandpass_Filter_Curve = np.loadtxt("/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/hsc_i.dat")
Y_Bandpass_Filter_Curve = np.loadtxt("/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/hsc_y.dat")
Z_Bandpass_Filter_Curve = np.loadtxt("/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/hsc_z.dat")


In [7]:
SED_File

Z_Redshift_Range,Z,Flux_SersicNOne_GaussianFixedSigma,Flux_SersicNOne_GaussianFixedSigma_SD
bytes11,float64,float64[5],float64[5]
0.25<z<0.35,0.3,42.59511863429468 .. 165.2212371318903,2.044952982675851 .. 5.413806831998792
0.35<z<0.45,0.4,17.921450416628023 .. 78.29113661982416,0.9746910676512054 .. 2.90205448971469
0.45<z<0.55,0.5,16.757313996924562 .. 42.47620194569488,0.5422505141894419 .. 1.5230812009587393
0.55<z<0.65,0.6,16.018992628200767 .. 30.345859427534695,0.7286584140583396 .. 1.269315649242704
0.65<z<0.75,0.7,12.371222662983188 .. 26.45619759147928,0.8833035378373185 .. 1.044955159767964
0.75<z<0.85,0.8,14.299682620888118 .. 28.710207228785652,0.7126644786030074 .. 1.1224599675109745
0.85<z<0.96,0.905,15.34812466890436 .. 26.61218691117512,1.3436087979859672 .. 1.4194662395992612


In [8]:
Redshift_Array = SED_File["Z"]

Flux_025_035 = SED_File["Flux_SersicNOne_GaussianFixedSigma"][0]
Flux_035_045 = SED_File["Flux_SersicNOne_GaussianFixedSigma"][1]
Flux_045_055 = SED_File["Flux_SersicNOne_GaussianFixedSigma"][2]
Flux_055_065 = SED_File["Flux_SersicNOne_GaussianFixedSigma"][3]
Flux_065_075 = SED_File["Flux_SersicNOne_GaussianFixedSigma"][4]
Flux_075_085 = SED_File["Flux_SersicNOne_GaussianFixedSigma"][5]
Flux_085_096 = SED_File["Flux_SersicNOne_GaussianFixedSigma"][6]

Flux_SD_025_035 = SED_File["Flux_SersicNOne_GaussianFixedSigma_SD"][0]
Flux_SD_035_045 = SED_File["Flux_SersicNOne_GaussianFixedSigma_SD"][1]
Flux_SD_045_055 = SED_File["Flux_SersicNOne_GaussianFixedSigma_SD"][2]
Flux_SD_055_065 = SED_File["Flux_SersicNOne_GaussianFixedSigma_SD"][3]
Flux_SD_065_075 = SED_File["Flux_SersicNOne_GaussianFixedSigma_SD"][4]
Flux_SD_075_085 = SED_File["Flux_SersicNOne_GaussianFixedSigma_SD"][5]
Flux_SD_085_096 = SED_File["Flux_SersicNOne_GaussianFixedSigma_SD"][6]

In [9]:
G_Bandpass_Wave = G_Bandpass_Filter_Curve[:, 0]  # First column (index 0)
G_Bandpass_Troughput = G_Bandpass_Filter_Curve[:, 1]  # Second column (index 1)

R_Bandpass_Wave = R_Bandpass_Filter_Curve[:, 0]  # First column (index 0)
R_Bandpass_Troughput = R_Bandpass_Filter_Curve[:, 1]  # Second column (index 1)

I_Bandpass_Wave = I_Bandpass_Filter_Curve[:, 0]  # First column (index 0)
I_Bandpass_Troughput = I_Bandpass_Filter_Curve[:, 1]  # Second column (index 1)

Z_Bandpass_Wave = Z_Bandpass_Filter_Curve[:, 0]  # First column (index 0)
Z_Bandpass_Troughput = Z_Bandpass_Filter_Curve[:, 1]  # Second column (index 1)

Y_Bandpass_Wave = Y_Bandpass_Filter_Curve[:, 0]  # First column (index 0)
Y_Bandpass_Troughput = Y_Bandpass_Filter_Curve[:, 1]  # Second column (index 1)

In [10]:
"""
SED Fitting with Prospector using Leja et al. 2019 model
For HSC SSP G, R, I, Z, Y photometry
Adapted for your specific data structure
"""

import numpy as np
from astropy.table import Table
from prospect.models import priors, sedmodel
from prospect.sources import CSPSpecBasis
try:
    from sedpy.observate import Filter
except ImportError:
    from sedpy import observate
    Filter = observate.Filter
import prospect.io.read_results as reader
from prospect.fitting import fit_model
from dynesty.dynamicsampler import stopping_function, weight_function

# ============================================
# 1. LOAD YOUR DATA
# ============================================

# Load the SED data
SED_File = Table.read("/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/Table_Gal_Flux_MicroJansky_For_Alexa.fits", format='fits')

# Load filter curves
G_Bandpass_Filter_Curve = np.loadtxt("/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/hsc_g.dat")
R_Bandpass_Filter_Curve = np.loadtxt("/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/hsc_r.dat")
I_Bandpass_Filter_Curve = np.loadtxt("/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/hsc_i.dat")
Y_Bandpass_Filter_Curve = np.loadtxt("/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/hsc_y.dat")
Z_Bandpass_Filter_Curve = np.loadtxt("/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/hsc_z.dat")

# Extract redshifts
Redshift_Array = SED_File["Z"]

# Extract fluxes for different redshift bins (in microjanskies)
Flux_025_035 = SED_File["Flux_SersicNOne_GaussianFixedSigma"][0]
Flux_035_045 = SED_File["Flux_SersicNOne_GaussianFixedSigma"][1]
Flux_045_055 = SED_File["Flux_SersicNOne_GaussianFixedSigma"][2]
Flux_055_065 = SED_File["Flux_SersicNOne_GaussianFixedSigma"][3]
Flux_065_075 = SED_File["Flux_SersicNOne_GaussianFixedSigma"][4]
Flux_075_085 = SED_File["Flux_SersicNOne_GaussianFixedSigma"][5]
Flux_085_096 = SED_File["Flux_SersicNOne_GaussianFixedSigma"][6]

# Extract flux uncertainties
Flux_SD_025_035 = SED_File["Flux_SersicNOne_GaussianFixedSigma_SD"][0]
Flux_SD_035_045 = SED_File["Flux_SersicNOne_GaussianFixedSigma_SD"][1]
Flux_SD_045_055 = SED_File["Flux_SersicNOne_GaussianFixedSigma_SD"][2]
Flux_SD_055_065 = SED_File["Flux_SersicNOne_GaussianFixedSigma_SD"][3]
Flux_SD_065_075 = SED_File["Flux_SersicNOne_GaussianFixedSigma_SD"][4]
Flux_SD_075_085 = SED_File["Flux_SersicNOne_GaussianFixedSigma_SD"][5]
Flux_SD_085_096 = SED_File["Flux_SersicNOne_GaussianFixedSigma_SD"][6]

# Parse filter curves
G_Bandpass_Wave = G_Bandpass_Filter_Curve[:, 0]
G_Bandpass_Troughput = G_Bandpass_Filter_Curve[:, 1]
R_Bandpass_Wave = R_Bandpass_Filter_Curve[:, 0]
R_Bandpass_Troughput = R_Bandpass_Filter_Curve[:, 1]
I_Bandpass_Wave = I_Bandpass_Filter_Curve[:, 0]
I_Bandpass_Troughput = I_Bandpass_Filter_Curve[:, 1]
Z_Bandpass_Wave = Z_Bandpass_Filter_Curve[:, 0]
Z_Bandpass_Troughput = Z_Bandpass_Filter_Curve[:, 1]
Y_Bandpass_Wave = Y_Bandpass_Filter_Curve[:, 0]
Y_Bandpass_Troughput = Y_Bandpass_Filter_Curve[:, 1]

# Pivot wavelengths (in Angstroms)
G_Band_Wave = 4791.155332473396
R_Band_Wave = 6214.060657315306
I_Band_Wave = 7688.726218179237
Z_Band_Wave = 8903.034519679457
Y_Band_Wave = 9771.696773766653


# ============================================
# 2. CREATE CUSTOM FILTERS
# ============================================

def create_custom_filters():
    """
    Create custom Filter objects from your filter curves
    Filter expects data as a tuple (wavelength, transmission)
    """
    filters = []
    
    # G band - pass data as tuple
    filters.append(Filter(name='hsc_g_custom', 
                         data=(G_Bandpass_Wave, G_Bandpass_Troughput)))
    
    # R band
    filters.append(Filter(name='hsc_r_custom', 
                         data=(R_Bandpass_Wave, R_Bandpass_Troughput)))
    
    # I band
    filters.append(Filter(name='hsc_i_custom', 
                         data=(I_Bandpass_Wave, I_Bandpass_Troughput)))
    
    # Z band
    filters.append(Filter(name='hsc_z_custom', 
                         data=(Z_Bandpass_Wave, Z_Bandpass_Troughput)))
    
    # Y band
    filters.append(Filter(name='hsc_y_custom', 
                         data=(Y_Bandpass_Wave, Y_Bandpass_Troughput)))
    
    return filters


# ============================================
# 3. BUILD OBSERVATIONS
# ============================================

def build_obs(flux_array, flux_err_array, redshift, filters):
    """
    Build observation dictionary from your data
    
    Parameters:
    -----------
    flux_array : array
        Photometry in microjanskies [G, R, I, Z, Y]
    flux_err_array : array
        Photometric uncertainties in microjanskies
    redshift : float
        Spectroscopic or photometric redshift
    filters : list
        List of Filter objects
    """
    
    # Convert microjanskies to maggies (what Prospector uses)
    # 1 maggies = 3631 Jy, so 1 uJy = 1e-6/3631 maggies
    maggies = np.array(flux_array) * 1e-6 / 3631.0
    maggies_err = np.array(flux_err_array) * 1e-6 / 3631.0
    
    # Create mask for valid data
    phot_mask = np.isfinite(maggies) & (maggies > 0) & np.isfinite(maggies_err) & (maggies_err > 0)
    
    obs = {
        'filters': filters,
        'maggies': maggies,
        'maggies_unc': maggies_err,
        'phot_mask': phot_mask,
        'wavelength': None,  # For photometry only
        'spectrum': None,
        'unc': None,
        'redshift': redshift,
    }
    
    return obs


# ============================================
# 4. DEFINE LEJA 2019 MODEL
# ============================================

def build_model(object_redshift=0.5, fixed_metallicity=False):
    """
    Build the Leja et al. 2019 model
    This uses a flexible parametric SFH with continuity prior
    """
    
    model_params = []
    
    # --- Basic Parameters ---
    model_params.append({
        'name': 'zred',
        'N': 1,
        'isfree': False,
        'init': object_redshift,
        'units': '',
        'prior': priors.TopHat(mini=0.0, maxi=4.0)
    })
    
    model_params.append({
        'name': 'mass',
        'N': 1,
        'isfree': True,
        'init': 1e10,
        'units': r'M$_\odot$',
        'prior': priors.LogUniform(mini=1e8, maxi=1e12)
    })
    
    model_params.append({
        'name': 'logzsol',
        'N': 1,
        'isfree': not fixed_metallicity,
        'init': -0.5,
        'units': r'$\log (Z/Z_\odot)$',
        'prior': priors.TopHat(mini=-1.5, maxi=0.2)
    })
    
    # --- Dust Attenuation (Calzetti) ---
    model_params.append({
        'name': 'dust2',
        'N': 1,
        'isfree': True,
        'init': 0.3,
        'units': '',
        'prior': priors.TopHat(mini=0.0, maxi=2.0)
    })
    
    model_params.append({
        'name': 'dust_index',
        'N': 1,
        'isfree': False,
        'init': -0.7,
        'units': '',
        'prior': None
    })
    
    # --- Nebular Emission ---
    model_params.append({
        'name': 'add_neb_emission',
        'N': 1,
        'isfree': False,
        'init': True,
        'units': '',
        'prior': None
    })
    
    model_params.append({
        'name': 'add_neb_continuum',
        'N': 1,
        'isfree': False,
        'init': True,
        'units': '',
        'prior': None
    })
    
    # --- Star Formation History (Leja 2019 flexible continuity) ---
    model_params.append({
        'name': 'sfh',
        'N': 1,
        'isfree': False,
        'init': 3,  # 3 = continuity SFH
        'units': '',
        'prior': None
    })
    
    model_params.append({
        'name': 'logsfr_ratios',
        'N': 6,  # Number of time bins - 1
        'isfree': True,
        'init': np.zeros(6),
        'units': '',
        'prior': priors.StudentT(mean=np.zeros(6), scale=np.ones(6)*0.3, df=2)
    })
    
    model_params.append({
        'name': 'logmass',
        'N': 1,
        'isfree': True,
        'init': 10.0,
        'units': r'$\log(M_*/M_\odot)$',
        'prior': priors.TopHat(mini=8.0, maxi=12.0)
    })
    
    # --- IMF ---
    model_params.append({
        'name': 'imf_type',
        'N': 1,
        'isfree': False,
        'init': 1,  # 1 = Chabrier
        'units': '',
        'prior': None
    })
    
    return sedmodel.SedModel(model_params)


# ============================================
# 5. RUN THE FIT FOR A SINGLE GALAXY
# ============================================

def run_prospector_fit_single(galaxy_idx, redshift_bin_fluxes, redshift_bin_errors, 
                               redshift, output_file='hsc_fit'):
    """
    Run Prospector fit for a single galaxy
    
    Parameters:
    -----------
    galaxy_idx : int
        Index of the galaxy in your arrays
    redshift_bin_fluxes : array
        Flux array from your redshift bin (e.g., Flux_025_035)
    redshift_bin_errors : array
        Error array from your redshift bin (e.g., Flux_SD_025_035)
    redshift : float
        Redshift of the galaxy
    output_file : str
        Output filename
    """
    
    # Get photometry for this galaxy (assuming columns are G, R, I, Z, Y)
    phot_ujy = redshift_bin_fluxes[galaxy_idx]
    phot_ujy_err = redshift_bin_errors[galaxy_idx]
    
    # Create custom filters
    filters = create_custom_filters()
    
    # Build observations and model
    obs = build_obs(phot_ujy, phot_ujy_err, redshift, filters)
    model = build_model(object_redshift=redshift)
    sps = CSPSpecBasis(zcontinuous=1)
    
    # Set up nested sampling parameters
    fitting_kwargs = {
        'nlive_init': 400,
        'nlive_batch': 200,
        'nested_bound': 'multi',
        'nested_sample': 'rwalk',
        'nested_dlogz_init': 0.05,
        'nested_posterior_thresh': 0.05,
        'nested_maxcall': int(1e7)
    }
    
    # Run the fit
    output = fit_model(obs, model, sps, lnprobfn=None, **fitting_kwargs)
    
    # Save results
    from prospect.io import write_results
    write_results.write_hdf5(
        output['run_params'],
        output,
        model,
        obs,
        hfile=f'{output_file}.h5',
        sps=sps
    )
    
    print(f"Fit complete! Results saved to {output_file}.h5")
    return output


# ============================================
# 6. EXAMPLE USAGE
# ============================================

if __name__ == "__main__":
    # Example: Fit the first galaxy in the z=0.25-0.35 bin
    galaxy_index = 0
    redshift = 0.3  # Midpoint of 0.25-0.35
    
    result = run_prospector_fit_single(
        galaxy_idx=galaxy_index,
        redshift_bin_fluxes=Flux_025_035,
        redshift_bin_errors=Flux_SD_025_035,
        redshift=redshift,
        output_file=f'galaxy_{galaxy_index}_z025_035'
    )
    
    # Access results
    print("\n=== Fit Results ===")
    print(f"Log stellar mass: {result['chain'][:, 0].mean():.2f}")
    print(f"Dust attenuation: {result['chain'][:, 1].mean():.2f}")

NameError: name 'fsps' is not defined

In [ ]:
# Fit first galaxy in z=0.25-0.35 bin
result = run_prospector_fit_single(
    galaxy_idx=0,  # which galaxy in the array
    redshift_bin_fluxes=Flux_025_035,
    redshift_bin_errors=Flux_SD_025_035,
    redshift=0.3,  # use actual redshift from Redshift_Array if available
    output_file='galaxy_0_z025_035'
)




In [25]:
"""
SED Fitting with Bagpipes using Leja et al. 2019 model
For HSC SSP G, R, I, Z, Y photometry
Much simpler installation than Prospector!
"""

import numpy as np
from astropy.table import Table
import bagpipes as pipes

# ============================================
# 1. LOAD YOUR DATA
# ============================================

# Load the SED data
SED_File = Table.read("/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/Table_Gal_Flux_MicroJansky_For_Alexa.fits", format='fits')

# Load filter curves
G_Bandpass_Filter_Curve = np.loadtxt("/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/hsc_g.dat")
R_Bandpass_Filter_Curve = np.loadtxt("/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/hsc_r.dat")
I_Bandpass_Filter_Curve = np.loadtxt("/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/hsc_i.dat")
Y_Bandpass_Filter_Curve = np.loadtxt("/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/hsc_y.dat")
Z_Bandpass_Filter_Curve = np.loadtxt("/home/jovyan/work/stampede3/AGN-Black-Hole-Research/AGN-Black-Hole-Research-Full-Catalog/hsc_z.dat")

# Extract redshifts
Redshift_Array = SED_File["Z"]

# Extract fluxes for different redshift bins (in microjanskies)
Flux_025_035 = SED_File["Flux_SersicNOne_GaussianFixedSigma"][0]
Flux_035_045 = SED_File["Flux_SersicNOne_GaussianFixedSigma"][1]
Flux_045_055 = SED_File["Flux_SersicNOne_GaussianFixedSigma"][2]
Flux_055_065 = SED_File["Flux_SersicNOne_GaussianFixedSigma"][3]
Flux_065_075 = SED_File["Flux_SersicNOne_GaussianFixedSigma"][4]
Flux_075_085 = SED_File["Flux_SersicNOne_GaussianFixedSigma"][5]
Flux_085_096 = SED_File["Flux_SersicNOne_GaussianFixedSigma"][6]

# Extract flux uncertainties
Flux_SD_025_035 = SED_File["Flux_SersicNOne_GaussianFixedSigma_SD"][0]
Flux_SD_035_045 = SED_File["Flux_SersicNOne_GaussianFixedSigma_SD"][1]
Flux_SD_045_055 = SED_File["Flux_SersicNOne_GaussianFixedSigma_SD"][2]
Flux_SD_055_065 = SED_File["Flux_SersicNOne_GaussianFixedSigma_SD"][3]
Flux_SD_065_075 = SED_File["Flux_SersicNOne_GaussianFixedSigma_SD"][4]
Flux_SD_075_085 = SED_File["Flux_SersicNOne_GaussianFixedSigma_SD"][5]
Flux_SD_085_096 = SED_File["Flux_SersicNOne_GaussianFixedSigma_SD"][6]

# Parse filter curves
G_Bandpass_Wave = G_Bandpass_Filter_Curve[:, 0]
G_Bandpass_Troughput = G_Bandpass_Filter_Curve[:, 1]
R_Bandpass_Wave = R_Bandpass_Filter_Curve[:, 0]
R_Bandpass_Troughput = R_Bandpass_Filter_Curve[:, 1]
I_Bandpass_Wave = I_Bandpass_Filter_Curve[:, 0]
I_Bandpass_Troughput = I_Bandpass_Filter_Curve[:, 1]
Z_Bandpass_Wave = Z_Bandpass_Filter_Curve[:, 0]
Z_Bandpass_Troughput = Z_Bandpass_Filter_Curve[:, 1]
Y_Bandpass_Wave = Y_Bandpass_Filter_Curve[:, 0]
Y_Bandpass_Troughput = Y_Bandpass_Filter_Curve[:, 1]


# ============================================
# 2. ADD CUSTOM FILTERS TO BAGPIPES
# ============================================

def add_hsc_filters_to_bagpipes():
    """
    Add HSC custom filters to Bagpipes filter directory
    Bagpipes expects filters as 2-column ASCII files in its filter directory
    """
    import os
    
    # Get Bagpipes filter directory
    bagpipes_dir = os.path.dirname(pipes.__file__)
    filter_dir = os.path.join(bagpipes_dir, "filters")
    
    print(f"Saving filters to: {filter_dir}")
    
    # Save custom filters (no .txt extension - Bagpipes adds it)
    np.savetxt(os.path.join(filter_dir, "hsc_g_custom"), 
               np.column_stack([G_Bandpass_Wave, G_Bandpass_Troughput]))
    np.savetxt(os.path.join(filter_dir, "hsc_r_custom"), 
               np.column_stack([R_Bandpass_Wave, R_Bandpass_Troughput]))
    np.savetxt(os.path.join(filter_dir, "hsc_i_custom"), 
               np.column_stack([I_Bandpass_Wave, I_Bandpass_Troughput]))
    np.savetxt(os.path.join(filter_dir, "hsc_z_custom"), 
               np.column_stack([Z_Bandpass_Wave, Z_Bandpass_Troughput]))
    np.savetxt(os.path.join(filter_dir, "hsc_y_custom"), 
               np.column_stack([Y_Bandpass_Wave, Y_Bandpass_Troughput]))
    
    print("Filters saved successfully!")
    print("Filter files created:")
    for f in ['hsc_g_custom', 'hsc_r_custom', 'hsc_i_custom', 'hsc_z_custom', 'hsc_y_custom']:
        print(f"  - {os.path.join(filter_dir, f)}")
    
# Run this once to add filters
print("Adding HSC filters to Bagpipes...")
add_hsc_filters_to_bagpipes()


# ============================================
# 3. PREPARE GALAXY DATA FOR BAGPIPES
# ============================================

def prepare_galaxy_data(flux_array, flux_err_array, redshift):
    """
    Prepare galaxy data in Bagpipes format
    
    Parameters:
    -----------
    flux_array : array
        Flux array with 5 elements [G, R, I, Z, Y] in microjanskies
    flux_err_array : array
        Error array with 5 elements in microjanskies
    redshift : float
        Galaxy redshift
    
    Returns:
    --------
    galaxy : dict
        Dictionary with photometry in Bagpipes format
    """
    
    # flux_array is already [G, R, I, Z, Y] for a single galaxy
    fluxes_ujy = flux_array
    errors_ujy = flux_err_array
    
    # Convert microjanskies to ergs/s/cm^2/Hz (Bagpipes format)
    # 1 uJy = 1e-29 erg/s/cm^2/Hz
    fluxes = fluxes_ujy * 1e-29
    errors = errors_ujy * 1e-29
    
    # Create photometry dictionary
    # Format: (flux, error) for each filter
    galaxy = {
        'hsc_g_custom': (fluxes[0], errors[0]),
        'hsc_r_custom': (fluxes[1], errors[1]),
        'hsc_i_custom': (fluxes[2], errors[2]),
        'hsc_z_custom': (fluxes[3], errors[3]),
        'hsc_y_custom': (fluxes[4], errors[4]),
    }
    
    return galaxy


# ============================================
# 4. DEFINE LEJA 2019 MODEL
# ============================================

def leja_2019_model():
    """
    Define the Leja et al. 2019 continuity SFH model
    This is the parametric model with flexible SFH
    """
    
    # Continuity SFH with 6 bins (Leja 2019 default)
    sfh = {}
    sfh["sfh"] = "continuity"  # Leja+2019 flexible parametric SFH
    
    # Mass and metallicity
    sfh["massformed"] = (8., 13.)  # Log stellar mass formed (solar masses)
    sfh["metallicity"] = (0.1, 2.5)  # Metallicity in Z/Z_sun
    
    # Dust attenuation
    dust = {}
    dust["type"] = "Calzetti"
    dust["Av"] = (0., 2.0)  # V-band attenuation
    
    # Combine into fit instructions
    fit_instructions = {}
    fit_instructions["sfh"] = sfh
    fit_instructions["dust"] = dust
    fit_instructions["nebular"] = {}  # Use default nebular emission
    
    return fit_instructions


# ============================================
# 5. RUN THE FIT
# ============================================

def run_bagpipes_fit(flux_array, flux_err_array, redshift, 
                     output_dir='bagpipes_output', run_name='galaxy'):
    """
    Run Bagpipes SED fit for a single galaxy
    
    Parameters:
    -----------
    flux_array : array
        Flux array with 5 elements [G, R, I, Z, Y] in microjanskies
    flux_err_array : array
        Error array with 5 elements in microjanskies
    redshift : float
        Galaxy redshift
    output_dir : str
        Directory for output files
    run_name : str
        Name for this fit
    """
    
    # Prepare galaxy data
    galaxy_data = prepare_galaxy_data(flux_array, flux_err_array, redshift)
    
    # Define filter list with full paths
    import os
    bagpipes_dir = os.path.dirname(pipes.__file__)
    filter_dir = os.path.join(bagpipes_dir, "filters")
    
    filter_list = [
        os.path.join(filter_dir, 'hsc_g_custom'),
        os.path.join(filter_dir, 'hsc_r_custom'),
        os.path.join(filter_dir, 'hsc_i_custom'),
        os.path.join(filter_dir, 'hsc_z_custom'),
        os.path.join(filter_dir, 'hsc_y_custom')
    ]
    
    # Create a load_data function that Bagpipes expects
    # For photometry only, it should return (photometry_array, redshift)
    def load_data(ID):
        # Return photometry as 1D array alternating flux, error for each filter
        phot_array = np.array([
            galaxy_data['hsc_g_custom'][0], galaxy_data['hsc_g_custom'][1],
            galaxy_data['hsc_r_custom'][0], galaxy_data['hsc_r_custom'][1],
            galaxy_data['hsc_i_custom'][0], galaxy_data['hsc_i_custom'][1],
            galaxy_data['hsc_z_custom'][0], galaxy_data['hsc_z_custom'][1],
            galaxy_data['hsc_y_custom'][0], galaxy_data['hsc_y_custom'][1]
        ])
        return phot_array, redshift
    
    # Define galaxy object with the load_data function and filter list
    galaxy = pipes.galaxy(run_name, load_data, 
                         photometry_exists=True,
                         filt_list=filter_list)
    
    # Get fit instructions
    fit_instructions = leja_2019_model()
    
    # Set up the fit
    fit = pipes.fit(galaxy, fit_instructions, run=output_dir)
    
    # Run nested sampling (MultiNest)
    fit.fit(verbose=True, n_live=400)
    
    print(f"\nFit complete for {run_name}!")
    print(f"Results saved to {output_dir}/{run_name}/")
    
    # Print best-fit parameters
    print("\n=== Best-fit parameters ===")
    print(f"Log stellar mass: {fit.fitted_model.sfh.stellar_mass:.2f}")
    print(f"Dust Av: {fit.fitted_model.dust.Av:.2f}")
    print(f"Metallicity (Z/Z_sun): {fit.fitted_model.sfh.metallicity:.2f}")
    
    return fit


# ============================================
# 6. PLOTTING RESULTS
# ============================================

def plot_fit_results(fit, run_name):
    """
    Create standard Bagpipes plots
    """
    import matplotlib.pyplot as plt
    
    # Plot SED fit
    fig = fit.plot_1d_posterior()
    plt.savefig(f'{run_name}_posterior.png', dpi=300, bbox_inches='tight')
    
    # Plot SED
    fig = fit.plot_sfh()
    plt.savefig(f'{run_name}_sfh.png', dpi=300, bbox_inches='tight')
    
    print(f"Plots saved as {run_name}_posterior.png and {run_name}_sfh.png")


# ============================================
# 7. EXAMPLE USAGE
# ============================================

if __name__ == "__main__":
    # Fit galaxy in z=0.25-0.35 bin
    # Each Flux_XXX_XXX array represents ONE galaxy with 5 filter measurements
    redshift = 0.3  # Midpoint of bin
    
    fit = run_bagpipes_fit(
        flux_array=Flux_025_035,  # [G, R, I, Z, Y] for this galaxy
        flux_err_array=Flux_SD_025_035,
        redshift=redshift,
        output_dir='hsc_bagpipes_fits',
        run_name='galaxy_z025_035'
    )
    
    # Plot results
    plot_fit_results(fit, 'galaxy_z025_035')
    




    

Adding HSC filters to Bagpipes...
Saving filters to: /home/jovyan/.local/lib/python3.12/site-packages/bagpipes/filters
Filters saved successfully!
Filter files created:
  - /home/jovyan/.local/lib/python3.12/site-packages/bagpipes/filters/hsc_g_custom
  - /home/jovyan/.local/lib/python3.12/site-packages/bagpipes/filters/hsc_r_custom
  - /home/jovyan/.local/lib/python3.12/site-packages/bagpipes/filters/hsc_i_custom
  - /home/jovyan/.local/lib/python3.12/site-packages/bagpipes/filters/hsc_z_custom
  - /home/jovyan/.local/lib/python3.12/site-packages/bagpipes/filters/hsc_y_custom


ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 5 and the array at index 1 has size 1

In [21]:
import os
bagpipes_dir = os.path.dirname(pipes.__file__)
filter_dir = os.path.join(bagpipes_dir, "filters")
print(os.listdir(filter_dir))


['__pycache__', 'UVJ', 'hsc_z_custom.txt', 'UVJ.filt_list', 'hsc_r_custom', 'hsc_r_custom.txt', 'hsc_y_custom', 'hsc_i_custom.txt', '__init__.py', 'filter_set.py', 'hsc_g_custom', 'hsc_z_custom', 'hsc_i_custom', 'hsc_y_custom.txt', 'hsc_g_custom.txt']
